# Resenas de e-commerce: sentimiento y prioridades de mejora

**Que responde este cuaderno**

1. Que parte del total de resenas es negativa.
2. Si el sentimiento calculado con VADER coincide con el rating que dejo el cliente.
3. Que productos conviene priorizar (mucho volumen y mucha negatividad).
4. Cuanto explica el sentimiento del rating.

**Datos**: `reviews_sentiment_TRANSFORMADO_ROBUSTO_100k.csv` (100.000 resenas), versionado en este repositorio. Se descarga por URL al ejecutar, no hace falta subir nada.

**Como correrlo**: Runtime > Run all. Solo usa pandas y matplotlib.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/Customer-Sentiment-Analysis-Product-Improvement-Drivers/main/'
CSV = 'reviews_sentiment_TRANSFORMADO_ROBUSTO_100k.csv'
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(BASE + CSV) as r:
    df = pd.read_csv(io.BytesIO(r.read()))

print('filas:', len(df), '| columnas:', list(df.columns))
df.head()

## 1. Cuanta resena negativa hay

Sin este numero no se puede priorizar nada: 8.000 resenas negativas sobre 100.000 no es lo mismo que sobre 10.000.

In [ ]:
total = len(df)
conteo = df['sentiment_category'].value_counts()
cuota = (conteo / total * 100).round(2)

tabla = pd.DataFrame({'resenas': conteo, 'pct': cuota})
print(tabla.to_string())

neg = int(conteo.get('Negative', 0))
print()
print(f'RESPUESTA: {cuota.get("Negative", 0)} % de las resenas ({neg} de {total}) es negativa.')

ax = cuota.plot(kind='bar', figsize=(6, 3))
ax.set_ylabel('% de resenas')
ax.set_title('Distribucion de sentimiento')
plt.tight_layout()
plt.savefig('outputs/sentiment_distribution.png', dpi=150)
plt.show()

## 2. Coincide VADER con el rating del cliente

El rating es la verdad del cliente; el sentimiento es lo que deduce el modelo del texto. Si no coinciden, uno de los dos mide otra cosa.

In [ ]:
cruce = pd.crosstab(df['rating_category'], df['sentiment_category'])
print(cruce.to_string())

acuerdo = ((df['rating'] >= 4) == (df['sentiment_category'] == 'Positive')).mean() * 100
solo5 = df[(df['rating'] == 5) & (df['sentiment_category'] == 'Negative')]
solo1 = df[(df['rating'] == 1) & (df['sentiment_category'] == 'Positive')]

print()
print(f'RESPUESTA: coinciden en signo en el {acuerdo:.1f} % de las resenas.')
print(f'Casos raros: {len(solo5)} resenas de 5 estrellas con texto negativo y {len(solo1)} de 1 estrella con texto positivo.')
print('Cuando el texto y la estrella no van juntos, la estrella suele ser mas fiable para negocio y el texto explica el motivo.')

## 3. Que productos priorizar

Un producto con 40 % de negatividad sobre 12 resenas no es una senal; sobre 300 si. Filtramos por volumen minimo y ordenamos por proporcion, no por cantidad absoluta.

In [ ]:
por_producto = df.groupby('product_id').agg(
    resenas=('Id', 'count'),
    negativas=('sentiment_category', lambda s: int((s == 'Negative').sum())),
    rating_medio=('rating', 'mean'),
)
por_producto['pct_negativas'] = (por_producto['negativas'] / por_producto['resenas'] * 100).round(1)
por_producto['rating_medio'] = por_producto['rating_medio'].round(2)

MIN_RESENAS = 30
candidatos = por_producto[por_producto['resenas'] >= MIN_RESENAS]
candidatos = candidatos.sort_values(['pct_negativas', 'negativas'], ascending=False)

print(candidatos.head(10).to_string())
candidatos.to_csv('outputs/prioridad_productos.csv')

print()
print(f'RESPUESTA: {len(candidatos)} productos tienen {MIN_RESENAS} o mas resenas.')
if len(candidatos):
    peor = candidatos.iloc[0]
    print(f'El peor es {candidatos.index[0]}: {peor.pct_negativas} % de resenas negativas ({int(peor.negativas)} de {int(peor.resenas)}).')

ax = candidatos.head(10)['pct_negativas'].sort_values().plot(kind='barh', figsize=(7, 4))
ax.set_xlabel('% de resenas negativas')
ax.set_title('Productos con mas negatividad (min. 30 resenas)')
plt.tight_layout()
plt.savefig('outputs/top_negativos.png', dpi=150)
plt.show()

## 4. Cuanto explica el sentimiento del rating

Una correlacion alta significa que el texto y la estrella miden lo mismo; una baja significa que el texto aporta informacion que la estrella no captura.

In [ ]:
corr = df['compound_score'].corr(df['rating'])
media_por_rating = df.groupby('rating')['compound_score'].agg(['mean', 'count']).round(3)
print(media_por_rating.to_string())

print()
print(f'RESPUESTA: la correlacion entre compound_score y rating es {corr:.3f}.')
print('El compound medio sube de forma monotona con la estrella: el texto acompana al rating, no lo contradice.')

ax = media_por_rating['mean'].plot(kind='line', marker='o', figsize=(6, 3))
ax.set_xlabel('rating del cliente')
ax.set_ylabel('compound medio')
ax.set_title('Sentimiento medio por rating')
plt.tight_layout()
plt.savefig('outputs/sentimiento_por_rating.png', dpi=150)
plt.show()

## Limites de este analisis

- VADER es un lexico, no un modelo entrenado para este dominio: la ironia y el sarcasmo lo confunden.
- `product_id` es un identificador tecnico; sin el catalogo no se puede decir que producto es.
- La negatividad del texto no mide impacto en ventas: mide percepcion.
- El CSV ya trae el sentimiento calculado; este cuaderno lo analiza, no lo recalcula.